# Itinera · Fine-tuning LoRA en Google Colab

Este experimento adapta un modelo pequeño para que responda con el estilo y la estructura de Itinera. No pretende memorizar precios ni información actual: esos datos siguen llegando desde la búsqueda web y las APIs. El objetivo del ajuste es aprender **formato, tono y criterios de planificación**.

> Activa una GPU en Colab: `Entorno de ejecución → Cambiar tipo de entorno → T4`. La muestra es deliberadamente pequeña para poder ejecutarla en una práctica.

In [ ]:
!pip -q install -U transformers datasets peft trl accelerate

## 1. Dataset sintético y revisión de una muestra
Generamos 48 ejemplos conversacionales. Cada respuesta separa viaje/alojamiento del presupuesto diario, agrupa zonas y añade una advertencia de verificación. En un proyecto real revisaría una muestra mayor y mezclaría ejemplos redactados por expertos.

In [ ]:
from datasets import Dataset
from itertools import product

cities = {
    'Lisboa': ('Alfama y Baixa', 'Belém', 'miradores'),
    'Sevilla': ('Santa Cruz y centro', 'Triana', 'plazas y patios'),
    'Roma': ('Centro Storico', 'Trastevere', 'parques y plazas'),
    'París': ('Marais y centro', 'Montmartre', 'jardines'),
    'Kioto': ('Higashiyama', 'Arashiyama', 'senderos y jardines'),
    'Oporto': ('Ribeira', 'Cedofeita', 'miradores del Duero'),
}
profiles = [
    ('cultura y gastronomía', 75, 'equilibrado'),
    ('historia y vida local', 50, 'ajustado'),
    ('naturaleza y fotografía', 95, 'cómodo'),
    ('arte y paseos', 65, 'tranquilo'),
]
durations = [2, 3]

def answer(city, zones, interests, budget, pace, days):
    z1, z2, free = zones
    lines = [
        f'PROPUESTA · {city} · {days} días',
        f'Enfoque: {interests}; ritmo {pace}.',
        f'Viaje + alojamiento: comparar directamente con operadores y alojarse en una zona conectada con {z1}.',
        f'Actividades + comida + transporte local: objetivo de {budget} € por persona y día.',
    ]
    for d in range(1, days + 1):
        zone = z1 if d == 1 else z2
        lines.append(f'Día {d} · {zone}: visita principal por la mañana, comida local, paseo por {free} y tarde flexible.')
    lines += ['Consejo: agrupa las paradas por barrios y deja margen entre actividades.',
              'Aviso: confirma precios, horarios y disponibilidad en las fuentes oficiales antes de reservar.']
    return '\n'.join(lines)

rows = []
for city, (interests, budget, pace), days in product(cities, profiles, durations):
    prompt = f'Organiza {days} días en {city}. Intereses: {interests}. Presupuesto local: {budget} € al día. Ritmo: {pace}.'
    rows.append({'messages': [
        {'role': 'system', 'content': 'Eres Itinera. Diseñas viajes claros, realistas y prudentes, sin inventar datos actuales.'},
        {'role': 'user', 'content': prompt},
        {'role': 'assistant', 'content': answer(city, cities[city], interests, budget, pace, days)},
    ]})
dataset = Dataset.from_list(rows).train_test_split(test_size=0.125, seed=42)
print(dataset)
print(dataset['train'][0]['messages'][-1]['content'])

## 2. Modelo base y prueba anterior al entrenamiento

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen3-0.6B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto'
)

test_messages = [
    {'role': 'system', 'content': 'Eres Itinera. Diseñas viajes claros, realistas y prudentes.'},
    {'role': 'user', 'content': 'Organiza 2 días en Lisboa, con cultura y comida, 60 € por día.'},
]
def generate(messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=260, do_sample=False)
    return tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

before = generate(test_messages)
print('ANTES DEL AJUSTE\n', before)

## 3. Entrenamiento LoRA
LoRA mantiene congelada la mayor parte del modelo y entrena adaptadores pequeños. Es mucho más asequible que un ajuste completo y el resultado se guarda en pocos megabytes.

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    task_type='CAUSAL_LM',
)
training_args = SFTConfig(
    output_dir='/content/itinera-lora',
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    logging_steps=2,
    eval_strategy='epoch',
    save_strategy='epoch',
    max_length=768,
    fp16=True,
    report_to='none',
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.train()
trainer.save_model('/content/itinera-lora/final')

## 4. Comparación cualitativa
La evaluación es intencionadamente sencilla: pérdida sobre el conjunto de validación y comparación de una muestra antes/después. Revisamos si respeta la estructura, separa presupuestos, evita afirmaciones actuales no verificadas y mantiene un tono útil.

In [ ]:
metrics = trainer.evaluate()
after = generate(test_messages)
print('Pérdida de validación:', round(metrics['eval_loss'], 4))
print('\nANTES\n', before)
print('\nDESPUÉS\n', after)
print('\nAdaptador guardado en /content/itinera-lora/final')

## Conclusión
El LoRA demuestra la fase de fine-tuning pedida en el caso. La aplicación desplegada usa el modelo alojado por API porque es más estable para una demo pública; el adaptador local se centra en enseñar el método y en comprobar si el modelo aprende el formato de Itinera.